
> # **1. RNN의 한계와 Transformer의 등장**

## 1-1. RNN의 한계
- RNN은 이전 시점의 **은닉 상태(Hidden State)**를 다음 시점에 전달하여 순차 데이터를 처리하는 신경망
- 이전 단어의 정보를 기억하며 다음 단어를 처리하는 방식
- But, 순차적으로 정보를 전달하는 구조로 인해 여러 한계 발생

### **1) 장기 의존성 문제**
: 시퀀스가 길어질수록 초반부의 중요한 정보가 후반부까지 충분히 전달되지 못하는 문제

- 이전 시점의 정보를 계속 전달하는 과정에서 초기 정보가 약해질 수 있음
- **기울기 소실(Vanishing Gradient)**: 역전파 과정에서 기울기가 반복적으로 곱해져 0에 가까워지는 현상
- **기울기 폭주(Exploding Gradient)**: 반대로 기울기가 지나치게 커져 학습이 불안정해지는 현상

### **2) 순차적 연산으로 인한 병렬 처리의 한계**
- RNN은 이전 시점의 계산 결과가 있어야 다음 시점의 연산 가능
- 따라서 같은 시퀀스 내 모든 단어를 동시에 처리하기 어려움
-> 학습 속도 저하 및 긴 시퀀스 처리의 비효율성 발생

### **3) LSTM과 GRU의 한계**
- LSTM과 GRU는 게이트를 통해 중요한 정보를 선택적으로 기억하여 RNN의 장기 의존성 문제 완화
- But, 여전히 이전 시점의 계산 결과에 의존하는 **순차 처리 구조** 유지
=> 장기 기억의 개선만으로는 병렬 처리 문제 해결 불가능


## 1-2. Attention의 등장

### **Seq2Seq의 병목 현상**
- 기존 Seq2Seq는 인코더가 입력 문장의 정보를 하나의 **Context Vector**에 압축
- 디코더는 고정된 크기의 Context Vector를 바탕으로 출력 문장 생성
- 입력 문장이 길어져도 동일한 크기의 벡터에 모든 정보를 담아야 함
-> 문장이 길수록 정보 손실 발생 가능성 증가

### **Attention의 개념**
: 출력 단어를 생성할 때 입력 문장의 모든 부분을 참고하되, **관련성이 높은 정보에 더 큰 가중치를 부여**하는 메커니즘

- 인코더의 마지막 Hidden State만 사용하는 대신 각 시점의 Hidden State를 모두 참고
- 디코더가 출력 단어를 생성할 때마다 입력 단어들의 관련도 계산
- 관련도에 따라 가중치를 부여하여 새로운 Context Vector 생성

ex) "I am a student"를 번역할 때
- "나는"을 생성할 때와 "학생"을 생성할 때 참고해야 하는 입력 정보가 다름
- "학생"을 생성할 때는 "student"에 높은 Attention Weight 부여

=> 출력 시점마다 필요한 정보를 선택적으로 활용하여 **Seq2Seq의 병목 현상 완화**

단, RNN 기반 Attention 모델은 인코더와 디코더가 여전히 순차적으로 작동하므로 병렬 처리의 한계 존재


## 1-3. Attention is All You Need

### **Transformer의 등장**
- 2017년 발표된 논문 *Attention Is All You Need*에서 Transformer 구조 제안
- 기존 RNN 및 CNN 기반의 순차 처리 없이 **Attention을 중심으로 시퀀스를 처리**하는 구조
- 입력 단어 간 관계를 직접 계산하여 문맥 파악
-> RNN의 순차적 계산을 제거함으로써 학습 과정의 병렬 처리 가능

### **Transformer란?**
: Self-Attention을 핵심 연산으로 사용하여 입력 문장의 관계를 파악하는 **Encoder-Decoder 기반 신경망**

* **Self-Attention**
     - 동일한 문장 내 단어들 사이의 관계를 계산
     - 문장 전체의 문맥적 의미 파악

* **Multi-Head Attention**
     - 여러 Attention을 동시에 계산
     - 서로 다른 관점에서 단어 간 관계 학습

* **Positional Encoding**
     - 단어의 위치 및 순서 정보를 별도로 추가
     - 순차적으로 처리하지 않는 구조의 한계 보완

**기존 모델과의 차이**
- RNN: 이전 단어의 정보를 다음 단어로 순차적으로 전달
- Seq2Seq + Attention: RNN 구조를 유지하면서 중요한 입력 정보에 집중
- Transformer: **Self-Attention으로 모든 단어 간 관계를 직접 계산**

=> 입력 시퀀스의 병렬 처리와 장거리 단어 관계 학습에 유리한 구조


> # **2. Self-Attention**

## 2-1. Query, Key, Value

### **Self-Attention의 기본 원리**
: 문장 내부의 각 단어가 다른 단어와 얼마나 관련 있는지 계산하고, 그 결과를 반영하여 새로운 벡터 생성

- Attention 계산을 위해 입력 벡터로부터 **Query, Key, Value** 생성
- 각 벡터는 서로 다른 학습 가능한 가중치 행렬을 통해 만들어짐

### **1) Query (Q)**
: 현재 단어가 다른 단어로부터 어떤 정보를 얻어야 하는지 나타내는 벡터

- 현재 단어가 필요한 정보를 찾기 위한 질문 역할
- 다른 단어들의 Key와 비교하여 관련도 계산

### **2) Key (K)**
: 각 단어가 어떤 정보를 가지고 있는지 나타내는 벡터

- Query와 비교되는 검색 기준 역할
- Query와의 관련도를 통해 Attention Score 결정

### **3) Value (V)**
: 각 단어가 실제로 전달하는 정보를 담은 벡터

- 계산된 Attention Weight를 적용하여 최종 결과 생성
- 관련도가 높은 단어의 정보일수록 결과 벡터에 더 크게 반영

**Attention의 계산 흐름**
1. Query와 Key를 비교하여 관련도 점수 계산
2. Softmax를 적용하여 Attention Weight 생성
3. Attention Weight와 Value의 가중합 계산
4. 문맥이 반영된 새로운 벡터 생성

=> **Q와 K는 어떤 정보를 참고할지 결정, V는 실제로 전달할 정보를 담당**


## 2-2. Cross-Attention vs. Self-Attention

### **Cross-Attention**
: 서로 다른 시퀀스 사이에서 필요한 정보를 참고하는 방식

- 일반적인 Encoder-Decoder 구조에서 디코더가 인코더의 출력 참고
- **Q는 Decoder**, **K와 V는 Encoder의 출력**에서 생성
- 디코더가 현재 출력 단어를 만들기 위해 입력 문장의 관련 정보 탐색

ex) "The cat is sleeping"을 번역하는 상황
- "자고"라는 단어를 생성할 때 원문의 "sleeping"에 높은 가중치 부여

### **Self-Attention**
: 하나의 시퀀스 내부에서 단어들이 서로를 참고하는 방식

- **Q, K, V 모두 동일한 입력 시퀀스**에서 생성
- 각 단어가 자신을 포함한 모든 단어와의 관련도 계산
- 단어 간 거리에 관계없이 직접적인 정보 교환 가능

ex) "가을 은행 냄새 지독해"
- "은행"이라는 단어의 의미를 파악하기 위해 "가을", "냄새" 등과의 관계 계산
- 문맥을 통해 금융기관보다 은행나무 열매의 의미를 반영한 벡터 형성 가능

### **두 방식의 차이**
- Cross-Attention: **서로 다른 시퀀스 사이의 관계** 학습
- Self-Attention: **동일한 시퀀스 내부의 관계** 학습

=> Transformer는 Encoder와 Decoder 내부에서 Self-Attention을 사용하고, Encoder와 Decoder 사이에서는 Cross-Attention 사용


## 2-3. Scaled Dot-Product Attention

### **Dot-Product Attention**
: Query와 Key 벡터의 **내적(Dot Product)**으로 관련도를 계산하는 방식

- 벡터의 각 차원에 해당하는 값들을 곱한 뒤 모두 더하여 점수 계산
- Q와 전치된 K를 행렬 곱하여 모든 단어 쌍의 Attention Score 계산
- Softmax를 통해 점수를 합이 1인 가중치 분포로 변환

**문제점**
- 벡터의 차원이 커질수록 내적값이 지나치게 커질 가능성 존재
- Softmax 결과가 특정 단어에 과도하게 집중할 수 있음
-> 기울기가 매우 작아져 학습이 불안정해질 수 있음

### **Scaled Dot-Product Attention**
: 기존 Dot-Product Attention에 **Scaling 과정**을 추가한 방식

$$
Attention(Q,K,V)=Softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

* **$QK^T$**
     - Query와 Key의 내적
     - 단어 간 관련도 점수 계산

* **$\sqrt{d_k}$**
     - Key 벡터 차원의 제곱근
     - 내적값의 크기를 조절하여 Softmax의 과도한 집중 완화

* **Softmax**
     - 관련도 점수를 합이 1인 Attention Weight로 변환

* **V**
     - 가중치를 적용해 실제 정보를 조합하여 최종 출력 생성

**전체 계산 과정**
1. Q와 K의 내적으로 Attention Score 계산
2. $\sqrt{d_k}$로 나누어 Scaling
3. Softmax를 통한 Attention Weight 계산
4. Attention Weight와 V를 곱하여 출력 벡터 생성

=> 단어 간 관련도를 안정적으로 계산하면서 문맥이 반영된 새로운 벡터 생성


> # **3. Multi-Head Attention**

## 3-1. Single-Head Attention VS. Multi-Head Attention

### **Single-Head Attention**
: 하나의 Attention Head를 이용해 단어 간 관계를 계산하는 방식

- 하나의 Q, K, V 가중치 행렬 세트 사용
- 하나의 Attention Weight 분포를 통해 정보 조합
- 여러 관계를 동시에 표현할 수 있지만 하나의 Attention 결과로 정보를 통합해야 한다는 한계 존재

### **Multi-Head Attention**
: 여러 개의 Attention Head가 서로 다른 관점에서 단어 간 관계를 동시에 계산하는 방식

- 각 Head는 서로 다른 가중치 행렬 사용
- Head마다 독립적인 Q, K, V 생성
- 각 Head의 Attention 결과를 결합하여 최종 출력 생성

ex) "Tom gave Mary a book yesterday"
- Head 1: 누가 주었는지 -> Tom과의 관계
- Head 2: 누구에게 주었는지 -> Mary와의 관계
- Head 3: 무엇을 주었는지 -> book과의 관계
- Head 4: 언제 주었는지 -> yesterday와의 관계

-> 실제로 Head의 역할이 미리 지정되는 것은 아니며 학습 과정에서 서로 다른 관계를 포착할 수 있음

**Multi-Head Attention의 장점**
1. 다양한 의미 및 문법적 관계를 서로 다른 관점에서 학습 가능
2. 여러 Attention 계산을 동시에 수행하여 병렬 처리 가능
3. 단일 Head보다 풍부한 문맥적 특징 표현 가능


## 3-2. Multi-Head Attention 작동 방식


### **1) Linear Projection**
- 입력 벡터에 Head별로 서로 다른 가중치 행렬 적용
- 각 Head에 대해 Q, K, V 생성
- 512차원의 입력을 Head마다 64차원으로 변환
-> 총 8개의 Q, K, V 세트 생성

### **2) Parallel Attention**
- 8개의 Head가 각각 Scaled Dot-Product Attention 계산
- 각 Head는 자신만의 Attention Weight를 통해 다른 정보에 집중
- Head당 64차원의 출력 벡터 생성

### **3) Concatenation**
- 8개 Head의 출력 벡터를 하나로 연결
- 64차원 × 8개 = 512차원
-> 여러 관점에서 추출한 정보를 하나의 벡터로 결합

### **4) Final Linear Projection**
- 결합된 벡터에 출력 가중치 행렬 $W^O$ 적용
- Head별로 추출한 정보를 통합하여 최종 벡터 생성

**전체 과정**
- 512차원 입력
-> Head별 64차원 Q, K, V 생성
-> 8개 Head의 Attention 병렬 계산
-> 출력 벡터 연결(Concat)
-> 최종 선형 변환

=> 서로 다른 관점의 정보를 통합하여 단어의 문맥적 표현을 강화하는 방식


> # **4. Transformer 전체 아키텍처**

## 4-1. 트랜스포머의 전처리 단계

Transformer는 입력 문장을 그대로 처리하지 않고, 모델이 이해할 수 있는 벡터 형태로 변환한 뒤 위치 정보를 추가해야 함

**전체 흐름**
: Text -> Tokenization -> Embedding -> Positional Encoding

### **1) Tokenization**
: 입력 문장을 모델이 처리할 수 있는 기본 단위인 **Token**으로 나누는 과정

- 단어, 문장부호 또는 단어의 일부가 토큰으로 사용될 수 있음
- 분리된 토큰을 토큰 사전의 정수 ID로 변환

ex) "I love you" -> ["I", "love", "you"]

### **2) Embedding**
: 토큰을 의미 정보를 학습할 수 있는 고정된 차원의 숫자 벡터로 변환하는 과정

- 각 토큰을 연속적인 벡터 공간에 표현
- 학습을 통해 의미 및 사용 맥락이 유사한 토큰이 비슷한 표현을 가질 수 있음
- Transformer 원 논문에서는 512차원 임베딩 사용

### **3) Positional Encoding**
: 단어들의 **순서 정보**를 임베딩 벡터에 추가하는 과정

- Transformer는 RNN과 달리 단어를 순서대로 처리하지 않음
- Self-Attention 연산만으로는 각 단어의 위치를 구분하기 어려움
- 따라서 위치별로 서로 다른 벡터를 만들어 토큰 임베딩에 더함

ex) "개가 고양이를 쫓는다"와 "고양이가 개를 쫓는다"
- 단어의 순서에 따라 문장 의미가 달라짐
- Positional Encoding을 통해 순서 정보 반영

=> **Embedding은 토큰의 의미, Positional Encoding은 토큰의 위치 정보를 제공**


## 4-2. Transformer의 Encoder와 Decoder


### **1) Encoder**
: 입력 문장 전체의 문맥과 단어 간 관계를 파악하는 부분



**Encoder Layer의 핵심 구성**

* **Multi-Head Self-Attention**
     - 입력 문장 내 모든 단어 간 관계 계산
     - 여러 Head를 통해 다양한 문맥적 특징 추출

* **Feed-Forward Network(FFN)**
     - Attention을 통해 생성된 각 토큰의 벡터를 추가적으로 가공
     - 각 토큰에 동일한 신경망을 독립적으로 적용
     - 원 논문 기준 512 -> 2048 -> 512차원의 변환

* **Add & Norm**
     - Residual Connection: 입력과 서브레이어 출력을 더해 정보와 기울기 전달을 도움
     - Layer Normalization: 벡터 값을 정규화하여 안정적인 학습 지원

**Encoder 출력**
- 입력 단어 각각에 문맥적 의미가 반영된 벡터 생성
- 최종 Encoder 출력을 Decoder의 Cross-Attention에서 활용


### **2) Decoder**
: Encoder가 파악한 입력 문장의 정보를 활용하여 출력 문장을 생성하는 부분

원 논문에서는 동일한 Decoder Layer를 **6개 반복**

**Decoder Layer의 핵심 구성**

* **Masked Multi-Head Self-Attention**
     - 현재까지 주어진 출력 토큰들 사이의 관계 계산
     - 미래 위치의 정보를 참고하지 못하도록 Mask 적용
     - 학습 과정에서 정답을 미리 참고하는 문제 방지

* **Encoder-Decoder Attention (Cross-Attention)**
     - Decoder의 Query와 Encoder 출력에서 생성한 Key, Value 사용
     - 입력 문장 중 현재 출력에 필요한 정보 선택적으로 참고

* **Feed-Forward Network**
     - Attention 결과를 토큰별로 추가 가공
     - Encoder와 동일하게 원 논문 기준 512 -> 2048 -> 512차원 변환

* **Add & Norm**
     - 각 서브레이어의 출력에 잔차 연결과 정규화 적용
     - 학습 안정성 및 정보 전달 개선

**최종 출력**
- Decoder의 마지막 출력을 Linear Layer에 전달
- Softmax를 통해 다음 토큰의 확률 분포 계산
- 다음 토큰을 생성하는 과정을 종료 토큰이 나올 때까지 반복


## 4-3. Transformer의 전체 작동 과정

ex) "I love you" -> "나는 너를 사랑해"

1. **입력 전처리**
     - 문장을 토큰화하고 각 토큰을 임베딩 벡터로 변환
     - Positional Encoding을 더해 위치 정보 추가

2. **Encoder**
     - Multi-Head Self-Attention으로 입력 단어 사이의 관계 계산
     - FFN을 통해 각 단어의 특징 가공
     - 입력 문장 전체의 문맥 정보 생성

3. **Decoder**
     - 시작 토큰을 입력받아 출력 문장 생성 시작
     - Masked Self-Attention으로 현재까지의 출력 문맥 파악
     - Cross-Attention을 통해 Encoder에서 필요한 정보 참고

4. **출력 생성**
     - Linear + Softmax로 다음 토큰의 확률 계산
     - 예측된 토큰을 바탕으로 다음 토큰 생성 반복
     - 종료 토큰이 생성되면 문장 생성 종료

**최종 정리**
- RNN: 은닉 상태를 통해 과거 정보를 순차적으로 전달
- Attention: 현재 필요한 정보에 선택적으로 집중
- Self-Attention: 동일한 문장 내 모든 단어의 관계 계산
- Multi-Head Attention: 다양한 관계를 여러 관점에서 동시에 학습
- Transformer: **Attention 중심의 Encoder-Decoder 구조로 시퀀스의 문맥을 효과적으로 학습**

=> **RNN의 순차 처리 한계를 극복하고 Attention을 핵심 구조로 활용한 모델**
